# Rich bricks — Tier A/B/C experiment grid (sharded)

Self-contained, **sharded** Kaggle runner: it clones the repository at a **pinned branch**,
precomputes/caches GPSE encodings for whichever datasets need them, runs ONE SHARD of the
Tier A/B/C grid (`configs/`) × `gamma ∈ {0, best}` × datasets × seeds, measures
oversquashing **before and after** the lifting on every run, and zips the results for
download. Nothing has to be edited after upload except the config cell below — run this
same notebook with a different `SHARD_ID` in several Kaggle sessions to cover the full grid;
results merge cleanly because `run_id` is unique and deterministic.

What the grid is testing (the scientific question of this branch):

* per-stage OFAT (Tier A), then interactions (Tier B), then the full rich stack (Tier C) —
  what each rich brick buys on its own, and in combination;
* the `gamma=0` / `gamma=best` crossing on EVERY config — does the OSq term still pay off
  once the bricks get richer, or did richer bricks make it redundant?
* the **synthetic bottleneck arm** is where an OSq-guided lifting is *supposed* to win — it
  is the falsification core, and it is never dropped by the runtime guard;
* `gamma = 0` never calls the proxy, so it reproduces the proxy-free objective exactly —
  that equality is asserted below rather than assumed.

Analysis happens locally on `results/runs.csv`; the plots at the end are sanity checks only.

In [ ]:
# =====================================================================
# CONFIG — every knob of this notebook lives in this cell.
# =====================================================================
REPO_URL = "https://github.com/AlGoRythm3000/Differentiable-Motif-Discovery.git"
BRANCH   = "feat/rich-bricks"        # pinned on purpose: results stay attributable
REPO_DIR = "/kaggle/working/repo"
OUT_DIR  = "/kaggle/working/results"

# --- sharding ---------------------------------------------------------
# Run this same notebook with a different SHARD_ID per Kaggle session;
# the full plan is built once (deterministic, sorted by run_id) then
# filtered by hash(run_id) % NUM_SHARDS == SHARD_ID, so every session
# covers a disjoint slice and nothing is run twice.
SHARD_ID   = 0
NUM_SHARDS = 1

# --- grid axes ---------------------------------------------------------
CONFIGS_DIR = f"{REPO_DIR}/configs"    # Tier A/B/C declarative configs
DATASETS    = ["synthetic_bottleneck", "MUTAG", "PROTEINS", "IMDB-BINARY", "ENZYMES", "NCI1"]
SEEDS       = [0, 1, 2]                # three minimum: results are reported mean +- std
BEST_PROXY  = "r_bar"                  # the winning proxy from feat/osq-proxy's grid
BEST_GAMMA  = 0.1                      # the winning OSq weight from feat/osq-proxy's grid

# --- optimization ------------------------------------------------------
EPOCHS           = 200
PATIENCE         = 30              # early stopping on validation accuracy
LR               = 0.005
WEIGHT_DECAY     = 5e-4
BATCH_SIZE       = 32
HIDDEN_DIM       = 64
SPARSITY_WEIGHT  = 0.05            # never 0 with a positive gamma: the OSq term alone
                                   # is minimized by the complete graph

# --- OSq estimator -------------------------------------------------------
HUTCH_K     = 16                   # probes / CG right-hand sides
CG_TOL      = 1e-5
CG_MAXITER  = 100
OSQ_EPS     = 1e-4                 # Laplacian grounding: keeps a disconnected structure
                                   # finite and caps how bad a bottleneck may score
OSQ_SAMPLE_GRAPHS = 8              # test graphs the before/after measurement averages over

# --- GPSE (Stage 1 rich brick) ------------------------------------------
GPSE_CACHE_DIR = "/kaggle/working/gpse_cache"   # one precomputation per dataset, reused
                                                # across every s1='gpse' config/seed

# --- synthetic arm -------------------------------------------------------
SYNTHETIC_FAMILY  = "tree_neighbors_match"   # or "path_of_cliques_match"
SYNTHETIC_GRAPHS  = 600
SYNTHETIC_CLASSES = 4
SYNTHETIC_DEPTH   = 3

# --- runtime guard ---------------------------------------------------
# If the shard's plan does not fit, runs are cut in this order: Tier C
# entirely, then Tier B entirely, then (within what remains) largest
# gamma first, then NCI1, then ENZYMES. Never the synthetic arm, never
# a seed, never a gamma=0 twin without its gamma>0 pair.
TIME_BUDGET_S = 8.0 * 3600
INSTALL_DEPS  = True               # set False if the Kaggle image already has everything

In [ ]:
# =====================================================================
# SETUP - dependencies, clone at the pinned branch, import
# =====================================================================
import os, subprocess, sys

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_geometric", "networkx", "pyyaml"], check=False)
    # topomodelx/toponetx pin numpy<2 and pull in `pyg-nightly` - a second,
    # different distribution of the torch_geometric package. Resolving them
    # together with torch_geometric above lets pip downgrade numpy under
    # packages built against numpy>=2 and mix files from both PyG
    # distributions in site-packages, which is what produces
    # "partially initialized module 'torch_geometric' has no attribute
    # 'typing'" at import time. --no-deps installs the packages themselves
    # without letting their own resolver touch torch_geometric or numpy again;
    # both are lazy-imported only where actually used (models/message_passing.py).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "topomodelx", "toponetx"], check=False)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

COMMIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("branch:", BRANCH)
print("commit:", COMMIT_SHA)   # recorded in every single result row
print("shard  :", f"{SHARD_ID} / {NUM_SHARDS}")

In [ ]:
# =====================================================================
# ENVIRONMENT LOG
# =====================================================================
import torch

from tools.config_loader import load_all_configs
from tools.experiment_grid import GridConfig, build_plan, environment_info, run_grid, trim_plan
from tools.results_store import ResultsStore

ENV = environment_info(REPO_DIR)
ENV["branch"] = BRANCH
ENV["shard"] = f"{SHARD_ID}/{NUM_SHARDS}"
for key, value in ENV.items():
    print(f"{key:>18}: {value}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{'device':>18}: {DEVICE}")

ALL_CONFIGS = load_all_configs(CONFIGS_DIR)
print(f"{'pipeline configs':>18}: {len(ALL_CONFIGS)} loaded from {CONFIGS_DIR}")

In [ ]:
# =====================================================================
# GPSE CACHE STEP — precompute once per dataset that any s1='gpse'
# config in the grid actually uses, BEFORE the grid loop, which would
# otherwise recompute GPSE hundreds of
# times. Whichever encoder actually ran (real GPSE vs. the pse_explicit
# fallback, if the pretrained checkpoint can't be fetched - e.g. no
# internet on this Kaggle session) is recorded here and reused by
# tools/experiment_grid.py's own cache lookup.
# =====================================================================
from tools.gpse_cache import attach_gpse_cache
from tools.experiment_grid import load_arm

GPSE_DATASETS_NEEDED = sorted({
    d for d in DATASETS
    if any(c["s1"] == "gpse" for c in ALL_CONFIGS.values())
})

gpse_sources = {}
for dataset_name in GPSE_DATASETS_NEEDED:
    _, _, _, source = load_arm(
        dataset_name,
        GridConfig(data_root="/kaggle/working/datasets", gpse_cache_dir=GPSE_CACHE_DIR,
                   synthetic_family=SYNTHETIC_FAMILY, synthetic_graphs=SYNTHETIC_GRAPHS,
                   synthetic_classes=SYNTHETIC_CLASSES, synthetic_depth=SYNTHETIC_DEPTH),
        needs_gpse=True,
    )
    gpse_sources[dataset_name] = source
    print(f"{dataset_name:>22}: s1_encoder_actual = {source}")

ENV["gpse_sources"] = gpse_sources  # written into env.json below - never a silent degrade

In [ ]:
# =====================================================================
# PLAN — build the FULL grid, then keep only this shard's slice.
# hashlib (not the builtin hash(), which is randomized per process) so
# the same SHARD_ID always gets the same runs across sessions/restarts.
# =====================================================================
import hashlib

config = GridConfig(
    datasets=DATASETS, configs_dir=CONFIGS_DIR,
    best_proxy=BEST_PROXY, best_gamma=BEST_GAMMA, seeds=SEEDS,
    epochs=EPOCHS, patience=PATIENCE, lr=LR, weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE, hidden_dim=HIDDEN_DIM, sparsity_weight=SPARSITY_WEIGHT,
    hutch_k=HUTCH_K, cg_tol=CG_TOL, cg_maxiter=CG_MAXITER, osq_eps=OSQ_EPS,
    osq_sample_graphs=OSQ_SAMPLE_GRAPHS,
    synthetic_family=SYNTHETIC_FAMILY, synthetic_graphs=SYNTHETIC_GRAPHS,
    synthetic_classes=SYNTHETIC_CLASSES, synthetic_depth=SYNTHETIC_DEPTH,
    time_budget_s=TIME_BUDGET_S, device=DEVICE,
    data_root="/kaggle/working/datasets", gpse_cache_dir=GPSE_CACHE_DIR,
    commit_sha=COMMIT_SHA,
)

full_plan = build_plan(config)

def _shard_of(run_id: str) -> int:
    return int(hashlib.md5(run_id.encode()).hexdigest(), 16) % NUM_SHARDS

plan = [spec for spec in full_plan if _shard_of(spec.run_id) == SHARD_ID]

print(f"{len(full_plan)} runs in the full grid, {len(plan)} in shard {SHARD_ID}/{NUM_SHARDS}\n")
for spec in plan[:10]:
    print(" ", spec.run_id)
print("  ...")

store = ResultsStore(OUT_DIR)
store.write_env(ENV)

In [ ]:
# =====================================================================
# SANITY CHECK — gamma = 0 must reproduce the proxy-free objective exactly
# =====================================================================
# Cheap, and it is the one equality the whole grid rests on: if it ever fails,
# the gamma=0 baseline is not comparable to the gamma=best arm and no
# brick x OSq interaction below means anything.
import torch

from tools.losses import DMDLoss
from tools.osq_proxies import get_proxy

_logits = torch.randn(6, 3)
_target = torch.randint(0, 3, (6,))
_mask = torch.ones(6, dtype=torch.bool)
_structure = {"alpha": torch.rand(6), "selector_log_prob": None}

_plain = DMDLoss(sparsity_weight=SPARSITY_WEIGHT)(_logits, _target, _mask, _structure)
_gated = DMDLoss(sparsity_weight=SPARSITY_WEIGHT, osq_weight=0.0,
                 osq_fn=get_proxy(BEST_PROXY))(_logits, _target, _mask, _structure)
assert _gated.total.item() == _plain.total.item()
print(f"gamma = 0 equivalence holds for proxy={BEST_PROXY!r}")

In [ ]:
# =====================================================================
# RUN THE GRID (this shard only)
# =====================================================================
# Every run is wrapped: a failure is stored with its traceback in the row and the
# loop continues. Results are written as each run finishes, so a session that
# dies still leaves everything already completed - and re-running this cell
# resumes instead of redoing (an existing run_id is skipped, never overwritten).
summary = run_grid(config, store, plan=plan)

print("\n" + "=" * 60)
print(f"completed : {summary['completed']}")
print(f"failed    : {summary['failed']}")
print(f"skipped   : {summary['skipped']} (already stored)")
print(f"dropped   : {len(summary['dropped'])} (time budget)")
print(f"elapsed   : {summary['elapsed_s'] / 60:.1f} min")
if summary["dropped"]:
    print("\nruns the budget forced out:")
    for run_id in summary["dropped"][:20]:
        print("  ", run_id)

In [ ]:
# =====================================================================
# SAVE + ZIP for one-click download
# =====================================================================
archive = store.zip(f"/kaggle/working/rich_bricks_results_shard{SHARD_ID}.zip")
print("archive:", archive)
print("rows   :", len(store.read_runs()))
print("files  :", sorted(p.name for p in store.out_dir.iterdir()))

In [ ]:
# =====================================================================
# QUICK SANITY PLOTS (convenience only - real analysis happens locally,
# after merging every shard's runs.csv)
# =====================================================================
import matplotlib.pyplot as plt
import pandas as pd

runs = pd.read_csv(store.runs_path)
runs = runs[runs["status"] == "ok"].copy()
for column in ["test_acc", "gamma", "r_bar_before", "r_bar_after"]:
    runs[column] = pd.to_numeric(runs[column], errors="coerce")

# Mean +- std over seeds, per (tier, config_id, dataset, gamma). A single-seed
# number is never reported.
table = (runs.groupby(["tier", "config_id", "dataset", "gamma"])["test_acc"]
              .agg(["mean", "std", "count"]).reset_index())
display(table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for config_id, group in runs.groupby("config_id"):
    by_gamma = group.groupby("gamma")["test_acc"].mean()
    axes[0].plot(by_gamma.index, by_gamma.values, marker="o", label=config_id)
axes[0].set_xlabel("gamma")
axes[0].set_ylabel("test accuracy (mean over seeds/datasets)")
axes[0].set_title("brick x OSq interaction (this shard)")
axes[0].legend(fontsize=6, ncol=2)

# The claim of Phase 1, still true here: did the lifting actually reduce the
# measured mean effective resistance, now checked across every rich config too?
axes[1].scatter(runs["r_bar_before"], runs["r_bar_after"], c=runs["gamma"], cmap="viridis", s=18)
limit = float(runs[["r_bar_before", "r_bar_after"]].max().max())
axes[1].plot([0, limit], [0, limit], "k--", lw=1)
axes[1].set_xlabel("R_bar before lifting")
axes[1].set_ylabel("R_bar after lifting")
axes[1].set_title("measured oversquashing, before vs after")

plt.tight_layout()
plt.show()